# PawVision model training

Run the same modular pipeline as the command line. Prepare the dataset first with `src/preprocessing.py`.


## Production model: transfer learning

The app now uses a fine-tuned ImageNet backbone. Reproduce it from the project root:

```
python src/preprocessing.py --image-size 224 --destination dataset_224
python src/train_transfer.py
python src/evaluate.py --promote --allow-repeat-test
```

The cells below train the original scratch CNN, which is kept for comparison.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from model import build_model, make_dataset


In [ ]:
train_ds = make_dataset(str(ROOT / 'dataset' / 'train'), shuffle=True)
val_ds = make_dataset(str(ROOT / 'dataset' / 'validation'))
model = build_model()
callbacks = [
    __import__('tensorflow').keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    __import__('tensorflow').keras.callbacks.ModelCheckpoint(ROOT / 'models' / 'dogs_cats_model.keras', monitor='val_accuracy', save_best_only=True),
    __import__('tensorflow').keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)


In [ ]:
# Test only after training: python src/evaluate.py
# Evaluation artifacts and genuine test metrics are written to outputs/.
